In [1]:
%pip install numpy 
%pip install pandas 
%pip install torch 
%pip install scikit-learn
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Extract Data


In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [71]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from tqdm import tqdm

# === Paths ===
main_csv = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_ratings_original_updated.csv"
stats_folder = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables"
summary_save_path = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/player_stats_summaries"
os.makedirs(summary_save_path, exist_ok=True)

# === Load Data ===
df = pd.read_csv(main_csv)[:3000]  # Limit for test
df.dropna(inplace=True)

df["raw_name"] = df["Name"]
df.drop(
    columns=["Unnamed: 0", "Player URL", "Team Link", "fbref_url", "fbref_alltimestat"],
    inplace=True,
)


# === Clean Folder Name ===
def clean_folder_name(name, player_id):
    # name_clean = re.sub(
    #    r"[^\w\s]", "", name
    # )  # Remove special characters, preserve spaces
    return f"{name}_{player_id}"


# === Flatten Positions ===
def flatten_positions(pos_string):
    if pd.isnull(pos_string):
        return ["Unknown"]
    output = []
    parts = [p.strip() for p in pos_string.split(",") if p.strip()]
    for part in parts:
        match = re.match(r"^([A-Z]+)\s*\((.*?)\)$", part)
        if match:
            role, locs = match.groups()
            locs = re.findall(r"[A-Z]", locs.upper())
            output.append(role)
            output.extend([loc + role for loc in locs])
        else:
            output.append(part)
    return sorted(set(output)) or ["Unknown"]


df["Flat_Positions"] = df["Positions"].apply(flatten_positions)
df["Primary_Position"] = df["Flat_Positions"].apply(lambda x: x[0] if x else "Unknown")


# === Encode Categorical Features ===
cat_cols = ["Name", "Team", "Nationality", "Primary_Position"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# === Multi-hot Encode Position Roles ===
mlb = MultiLabelBinarizer()
multi_pos_df = pd.DataFrame(
    mlb.fit_transform(df["Flat_Positions"]), columns=mlb.classes_, index=df.index
)
df = pd.concat([df, multi_pos_df], axis=1)

# === Scale Age ===
df["Age_Original"] = df["Age"]
scaler_age = StandardScaler()
df["Age"] = scaler_age.fit_transform(df[["Age"]])
df["Age_Scaled"] = df["Age"]

# === Targets ===
target_cols = ["Rating", "Potential", "Value"]
y = df[target_cols].values


# === Load & Save Stats Per Player ===
def load_player_stats(folder_path, player_id, name, save_path):
    folder_name = clean_folder_name(name, player_id)
    full_path = os.path.join(folder_path, folder_name)

    if not os.path.isdir(full_path):
        print(f"Missing folder: {full_path}")
        return None

    summaries = []
    for file in os.listdir(full_path):
        if file.endswith(".csv"):
            file_path = os.path.join(full_path, file)
            try:
                data = pd.read_csv(file_path)
                summary = {}

                numeric = data.select_dtypes(include=np.number)
                for col in numeric.columns:
                    summary[f"{file}_{col}"] = numeric[col].mean()

                for meta_col in ["season", "club", "date", "team", "competition"]:
                    if meta_col in data.columns:
                        summary[f"{file}_{meta_col}"] = data[meta_col].mode().iloc[0]

                summaries.append(pd.Series(summary))
            except Exception as e:
                print(f"Error reading {file}: {e}")

    if summaries:
        final_summary = pd.DataFrame(summaries).mean(numeric_only=True).to_frame().T
        output_file = os.path.join(save_path, f"{player_id}_summary.csv")
        final_summary.to_csv(output_file, index=False)
        return output_file
    return None

In [72]:
print("Aggregating and saving per-player stats...")

missing_log = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    player_id = row["player_id"]
    player_name = row["raw_name"]

    result = load_player_stats(stats_folder, player_id, player_name, summary_save_path)
    if result is None:
        missing_log.append((player_id, player_name))

print("Done. Player summaries saved to:", summary_save_path)

# === Optional: Save Missing Folder Log ===
if missing_log:
    missing_df = pd.DataFrame(missing_log, columns=["PlayerID", "Name"])
    missing_df.to_csv("missing_folders.csv", index=False)
    print(f"Missing folders logged to missing_folders.csv ({len(missing_log)} entries)")

Aggregating and saving per-player stats...


  1%|          | 29/2983 [01:10<1:56:32,  2.37s/it]

Error reading stats_shooting_collapsed.csv: No columns to parse from file


100%|██████████| 2983/2983 [1:30:38<00:00,  1.82s/it]

Done. Player summaries saved to: C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/player_stats_summaries
Missing folders logged to missing_folders.csv (6 entries)


In [105]:
# === Load saved summaries and merge ===
summary_files = {
    os.path.splitext(file)[0]: os.path.join(summary_save_path, file)
    for file in os.listdir(summary_save_path)
    if file.endswith(".csv")
}

# Create list of stat dataframes in player order
stats_rows = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    player_id = str(row["player_id"])
    summary_path = summary_files.get(player_id)
    if summary_path:
        try:
            stats = pd.read_csv(summary_path)
            stats_rows.append(stats.iloc[0])  # First row from summary
        except Exception as e:
            print(f"Error loading {summary_path}: {e}")
            stats_rows.append(pd.Series(dtype=float))  # Fallback
    else:
        stats_rows.append(pd.Series(dtype=float))  # Missing summary

stats_df = pd.DataFrame(stats_rows).fillna(0)

# === Combine with original data ===
df_combined = pd.concat(
    [
        df.reset_index(drop=True),
        multi_pos_df.reset_index(drop=True),
        stats_df.reset_index(drop=True),
    ],
    axis=1,
)

df_combined.fillna(0, inplace=True)
# Remove duplicated columns based on name
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]
df_combined = df_combined.drop(columns=["D", "F", "M"])

# === Prepare feature matrix ===
X_df = df_combined.drop(columns=target_cols).copy()

# Encode any remaining object columns
for col in X_df.select_dtypes(include=["object"]).columns:
    if col in encoders:
        X_df[col] = encoders[col].transform(X_df[col].astype(str))
    else:
        print(f"⚠️ Unexpected string column removed: {col}")
        X_df.drop(columns=[col], inplace=True)

# Check again before scaling
object_cols = X_df.select_dtypes(include="object").columns.tolist()
if object_cols:
    print("⚠️ Dropping unexpected string columns:", object_cols)
    X_df.drop(columns=object_cols, inplace=True)

# Now it's safe to scale
X = X_df.values
# === Final scaling ===
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)


print("✅ Final preprocessing complete. Feature matrix shape:", X_scaled.shape)

100%|██████████| 2983/2983 [00:01<00:00, 2911.21it/s]


⚠️ Unexpected string column removed: Positions
⚠️ Unexpected string column removed: player_id
⚠️ Unexpected string column removed: raw_name
⚠️ Unexpected string column removed: Flat_Positions
✅ Final preprocessing complete. Feature matrix shape: (2983, 37)


Verifying columns


In [106]:
for i in df_combined.columns:
    print(i)

Name
Age
Team
Positions
Nationality
Rating
Potential
Value
player_id
raw_name
Flat_Positions
Primary_Position
AM
AMC
AML
AMR
Bayer U19
Bochum U19
Borussia U19
CAM
CD
CDM
CF
CM
DM
GK
Hertha U19
LAM
LD
LF
LM
Mainz U19
Paderborn U19
RAM
RD
RF
RM
Schalke U19
St. Pauli U19
Union U19
Werder U19
Wolfsburg U19
Age_Original
Age_Scaled


# Model 1 : Rating_per_position + general


In [107]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [108]:
def build_position_vocab(df):
    all_positions = sorted(
        set(pos for positions in df["Flat_Positions"] for pos in positions)
    )
    return {pos: idx for idx, pos in enumerate(all_positions)}


def expand_player_positions(df, numeric_cols, position_vocab):
    rows = []
    for _, row in df.iterrows():
        for pos in row["Flat_Positions"]:
            new_row = {col: row[col] for col in numeric_cols}
            new_row["player_id"] = row["player_id"]
            new_row["raw_name"] = row["raw_name"]
            new_row["position"] = pos
            new_row["position_index"] = position_vocab[pos]
            new_row["rating"] = row["Rating"]
            new_row["potential"] = row["Potential"]
            rows.append(new_row)
    return pd.DataFrame(rows)


In [109]:
class BottleneckFCBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout_rate=0.4, residual_scale=0.5):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.activation1 = nn.GELU()
        self.dropout1 = nn.Dropout(dropout_rate)

        self.fc2 = nn.Linear(hidden_dim, in_dim)
        self.norm2 = nn.LayerNorm(in_dim)
        self.activation2 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout_rate)
        self.residual_scale = residual_scale

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.norm1(out)
        out = self.activation1(out)
        out = self.dropout1(out)

        out = self.fc2(out)
        out = self.norm2(out)
        out = self.activation2(out)
        out = self.dropout2(out)

        return identity + self.residual_scale * out


class PositionEmbedder(nn.Module):
    def __init__(self, num_positions, embed_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(num_positions, embed_dim)
        self.linear = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.activation = nn.ReLU()

    def forward(self, position_indices):
        pos_embed = self.embedding(position_indices)
        pos_embed = self.linear(pos_embed)
        pos_embed = self.activation(pos_embed)
        return self.norm(pos_embed)


class FeatureProcessor(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            BottleneckFCBlock(hidden_dim, hidden_dim // 2),
            BottleneckFCBlock(hidden_dim, hidden_dim // 2),  # extra depth
        )

    def forward(self, x):
        return self.net(x)


class RatingPotentialModel(nn.Module):
    def __init__(self, input_dim, num_positions, hidden_dim=128):
        super().__init__()
        self.pos_embedder = PositionEmbedder(num_positions, embed_dim=16)
        self.num_processor = FeatureProcessor(input_dim, hidden_dim)

        self.cross_layer = nn.Sequential(
            nn.Linear(hidden_dim + 16, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.output_head = nn.Linear(hidden_dim, 2)  # Rating and Potential

    def forward(self, x_num, position_indices):
        pos_embed = self.pos_embedder(position_indices)  # [B, 16]
        x_num_proc = self.num_processor(x_num)  # [B, hidden]
        x = torch.cat([x_num_proc, pos_embed], dim=1)  # [B, hidden + 16]
        x = self.cross_layer(x)
        return self.output_head(x)

In [110]:
numeric_cols = [
    "CAM",
    "CD",
    "CDM",
    "CF",
    "CM",
    "DM",
    "LAM",
    "LD",
    "LF",
    "LM",
    "RAM",
    "RD",
    "RF",
    "RM",
]

position_vocab = build_position_vocab(df_combined)
expanded_df = expand_player_positions(df_combined, numeric_cols, position_vocab)

X_numeric = expanded_df[numeric_cols].values
X_position = expanded_df["position_index"].values
y = expanded_df[["rating", "potential"]].values

X_num_train, X_num_val, X_pos_train, X_pos_val, y_train, y_val = train_test_split(
    X_numeric, X_position, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_rp = MinMaxScaler(feature_range=(50, 100))
# scaler_rp = StandardScaler()
scaler_rp.fit(y_train)
y_train_scaled = scaler_rp.transform(y_train)
y_val_scaled = scaler_rp.transform(y_val)

In [116]:
class PlayerPositionDataset(torch.utils.data.Dataset):
    def __init__(self, X_num, X_pos, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_pos = torch.tensor(X_pos, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_pos[idx], self.y[idx]


train_dataset = PlayerPositionDataset(X_num_train, X_pos_train, y_train)
val_dataset = PlayerPositionDataset(X_num_val, X_pos_val, y_val)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32)

In [117]:
model = RatingPotentialModel(
    input_dim=X_numeric.shape[1], num_positions=len(position_vocab)
)
criterion = nn.HuberLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-5)


def train_model(
    model, train_loader, val_loader, max_epochs=1000, patience=40, min_delta=1e-2
):
    best_val_loss = float("inf")
    best_epoch = 0
    wait = 0
    epoch = 0
    history = []

    while epoch < max_epochs:
        epoch += 1
        model.train()
        total_loss = 0

        for X_num, X_pos, y in train_loader:
            X_num, X_pos, y = X_num, X_pos, y
            preds = model(X_num, X_pos)
            loss = criterion(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = sum(
                criterion(model(Xn, Xp), yb).item() for Xn, Xp, yb in val_loader
            ) / len(val_loader)

        history.append((epoch, avg_train_loss, val_loss))
        print(
            f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={val_loss:.4f}"
        )

        # Early stopping logic
        if val_loss + min_delta < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            wait = 0
            # torch.save(model.state_dict(), "best_model.pt")  # Optional checkpoint
        else:
            wait += 1
            if wait >= patience:
                print(
                    f"Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch}"
                )
                break

    return history

In [118]:
def evaluate_best_position(model, player_numeric_data, position_indices):
    model.eval()
    X_num = torch.tensor(player_numeric_data, dtype=torch.float32)
    X_pos = torch.tensor(position_indices, dtype=torch.long)
    with torch.no_grad():
        preds = model(X_num, X_pos)  # [num_positions, 2]
        best_idx = torch.argmax(preds[:, 1])
        return preds[best_idx].cpu().numpy(), best_idx.item()

In [119]:
train_model(model, train_loader, val_loader)

Epoch 1: Train Loss=66.7706, Val Loss=67.0143
Epoch 2: Train Loss=66.2272, Val Loss=66.1553
Epoch 3: Train Loss=65.0463, Val Loss=64.2986
Epoch 4: Train Loss=62.6675, Val Loss=60.8564
Epoch 5: Train Loss=58.5962, Val Loss=55.4651
Epoch 6: Train Loss=52.4418, Val Loss=47.7645
Epoch 7: Train Loss=43.9668, Val Loss=37.4935
Epoch 8: Train Loss=32.8468, Val Loss=24.4260
Epoch 9: Train Loss=19.2888, Val Loss=10.6706
Epoch 10: Train Loss=9.6908, Val Loss=6.2346
Epoch 11: Train Loss=7.6426, Val Loss=6.1116
Epoch 12: Train Loss=7.3493, Val Loss=6.0916
Epoch 13: Train Loss=7.3619, Val Loss=6.0750
Epoch 14: Train Loss=7.2979, Val Loss=6.0450
Epoch 15: Train Loss=7.2245, Val Loss=6.0664
Epoch 16: Train Loss=7.1402, Val Loss=5.9995
Epoch 17: Train Loss=7.2084, Val Loss=6.0432
Epoch 18: Train Loss=7.2756, Val Loss=5.9505
Epoch 19: Train Loss=7.1743, Val Loss=6.0068
Epoch 20: Train Loss=7.1784, Val Loss=5.9422
Epoch 21: Train Loss=7.2228, Val Loss=5.8730
Epoch 22: Train Loss=7.1728, Val Loss=5.9170
E

[(1, 66.77056645148963, 67.01433135986328),
 (2, 66.22718381642098, 66.15529113769531),
 (3, 65.04625065482442, 64.29856582641601),
 (4, 62.66752204703326, 60.85641098022461),
 (5, 58.596200300820506, 55.46505142211914),
 (6, 52.4418398028043, 47.76449821472168),
 (7, 43.9668046002412, 37.49351593017578),
 (8, 32.846808697111044, 24.4260298538208),
 (9, 19.288779565437356, 10.670556106567382),
 (10, 9.690815472722653, 6.234586038589478),
 (11, 7.642579196086481, 6.111631526947021),
 (12, 7.349342020312745, 6.091558561325074),
 (13, 7.361850932614887, 6.074970207214355),
 (14, 7.2979358141146715, 6.0450349617004395),
 (15, 7.224507894947301, 6.066391305923462),
 (16, 7.140160023866587, 5.999465074539184),
 (17, 7.208391632866021, 6.043188257217407),
 (18, 7.2756119828727375, 5.9505096054077145),
 (19, 7.174345419035485, 6.006821546554566),
 (20, 7.178372292063344, 5.942199974060059),
 (21, 7.22275878676218, 5.872987470626831),
 (22, 7.172769014559798, 5.9169652462005615),
 (23, 7.126362

In [120]:
def evaluate_players_per_position(df, model, numeric_cols, position_vocab, scaler_rp):
    model.eval()
    predictions = []

    for _, row in df.iterrows():
        exclude_positions = {"M", "F", "D", "AM"}  # Positions to ignore

        positions = [p for p in row["Flat_Positions"] if p not in exclude_positions]

        if not positions:
            continue

        player_features = np.array([row[col] for col in numeric_cols])
        features_expanded = np.tile(player_features, (len(positions), 1))
        position_indices = [position_vocab[pos] for pos in positions]

        X_num_tensor = torch.tensor(features_expanded, dtype=torch.float32)
        X_pos_tensor = torch.tensor(position_indices, dtype=torch.long)

        with torch.no_grad():
            preds_scaled = model(X_num_tensor, X_pos_tensor).cpu().numpy()
            preds_unscaled = scaler_rp.inverse_transform(
                preds_scaled
            )  # Now between 0–100

        best_idx = np.argmax(preds_unscaled[:, 1])  # Best potential
        best_rating, best_potential = preds_unscaled[best_idx]
        best_position = positions[best_idx]

        predictions.append(
            {
                "player_id": row["player_id"],
                "raw_name": row["raw_name"],
                "age": row["Age_Original"],
                "best_position": best_position,
                "pred_rating": best_rating,
                "pred_potential": best_potential,
            }
        )

    return pd.DataFrame(predictions)

In [121]:
pred_df = evaluate_players_per_position(
    df_combined, model, numeric_cols, position_vocab, scaler_rp
)

print("\n🔝 Best Position-Based Predictions:\n")
for _, row in pred_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Rating: {row['pred_rating']:.2f} | Potential: {row['pred_potential']:.2f}"
    )


🔝 Best Position-Based Predictions:

Erling Haaland            | Age: 24.0 | Position: CF   | Rating: 48.68 | Potential: 58.44
Lamine Yamal              | Age: 17.0 | Position: RAM  | Rating: 54.07 | Potential: 65.24
Kylian Mbappé             | Age: 26.0 | Position: CF   | Rating: 48.68 | Potential: 58.44
Pedri                     | Age: 22.0 | Position: DM   | Rating: 52.23 | Potential: 62.72
Jude Bellingham           | Age: 22.0 | Position: CM   | Rating: 52.37 | Potential: 63.24
Florian Wirtz             | Age: 22.0 | Position: CAM  | Rating: 50.42 | Potential: 60.56
Pau Cubarsí               | Age: 18.0 | Position: CD   | Rating: 52.35 | Potential: 63.23
Jamal Musiala             | Age: 22.0 | Position: CM   | Rating: 50.69 | Potential: 61.21
Alexander Isak            | Age: 25.0 | Position: CF   | Rating: 48.68 | Potential: 58.44
Cole Palmer               | Age: 23.0 | Position: CM   | Rating: 50.69 | Potential: 61.21
Federico Valverde         | Age: 26.0 | Position: DM   | Rating

In [122]:
def evaluate_scaled_predictions(df, model, numeric_cols, position_vocab):
    model.eval()
    scaled_preds = []

    for _, row in df.iterrows():
        exclude_positions = {"M", "F", "D", "AM"}  # Positions to ignore

        positions = [p for p in row["Flat_Positions"] if p not in exclude_positions]
        if not positions:
            continue

        player_features = np.array([row[col] for col in numeric_cols])
        features_expanded = np.tile(player_features, (len(positions), 1))
        position_indices = [position_vocab[pos] for pos in positions]

        X_num_tensor = torch.tensor(features_expanded, dtype=torch.float32)
        X_pos_tensor = torch.tensor(position_indices, dtype=torch.long)

        with torch.no_grad():
            preds_scaled = model(X_num_tensor, X_pos_tensor).cpu().numpy()

        best_idx = np.argmax(preds_scaled[:, 1])  # Max potential
        rating_scaled, potential_scaled = preds_scaled[best_idx]
        best_position = positions[best_idx]

        scaled_preds.append(
            {
                "player_id": row["player_id"],
                "raw_name": row["raw_name"],
                "age": row["Age_Original"],
                "best_position": best_position,
                "scaled_rating": rating_scaled,
                "scaled_potential": potential_scaled,
            }
        )

    return pd.DataFrame(scaled_preds)

In [123]:
scaled_df = evaluate_scaled_predictions(
    df_combined, model, numeric_cols, position_vocab
)

print("\n🔍 Scaled Position-Based Predictions:\n")
for _, row in scaled_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Scaled Rating: {row['scaled_rating']:.4f} | Scaled Potential: {row['scaled_potential']:.4f}"
    )


🔍 Scaled Position-Based Predictions:

Erling Haaland            | Age: 24.0 | Position: CF   | Scaled Rating: 62.3331 | Scaled Potential: 68.2296
Lamine Yamal              | Age: 17.0 | Position: RAM  | Scaled Rating: 67.0458 | Scaled Potential: 73.4257
Kylian Mbappé             | Age: 26.0 | Position: CF   | Scaled Rating: 62.3331 | Scaled Potential: 68.2296
Pedri                     | Age: 22.0 | Position: DM   | Scaled Rating: 65.4378 | Scaled Potential: 71.4989
Jude Bellingham           | Age: 22.0 | Position: CM   | Scaled Rating: 65.5594 | Scaled Potential: 71.8928
Florian Wirtz             | Age: 22.0 | Position: CAM  | Scaled Rating: 63.8533 | Scaled Potential: 69.8502
Pau Cubarsí               | Age: 18.0 | Position: CD   | Scaled Rating: 65.5389 | Scaled Potential: 71.8858
Jamal Musiala             | Age: 22.0 | Position: CM   | Scaled Rating: 64.0929 | Scaled Potential: 70.3466
Alexander Isak            | Age: 25.0 | Position: CF   | Scaled Rating: 62.3331 | Scaled Potentia

In [124]:
import torch
import torch.nn as nn


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(
            out_channels, out_channels * self.expansion, kernel_size=1, bias=False
        )
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


def make_layer(block, in_channels, out_channels, blocks, stride=1):
    downsample = None
    if stride != 1 or in_channels != out_channels * block.expansion:
        downsample = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels * block.expansion,
                kernel_size=1,
                stride=stride,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels * block.expansion),
        )

    layers = [block(in_channels, out_channels, stride, downsample)]
    for _ in range(1, blocks):
        layers.append(block(out_channels * block.expansion, out_channels))

    return nn.Sequential(*layers)


class CustomResNet50(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(
            3, self.in_channels, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.bn1 = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet50 uses [3, 4, 6, 3] Bottleneck blocks
        self.layer1 = make_layer(Bottleneck, 64, 64, blocks=3)
        self.layer2 = make_layer(Bottleneck, 256, 128, blocks=4, stride=2)
        self.layer3 = make_layer(Bottleneck, 512, 256, blocks=6, stride=2)
        self.layer4 = make_layer(Bottleneck, 1024, 512, blocks=3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * Bottleneck.expansion, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [125]:
class ResidualMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_rate=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, input_dim),
            nn.LayerNorm(input_dim),
            nn.GELU(),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        return x + 0.5 * self.block(x)  # Scaled residual


class DeepRegressionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.layers(x)


In [126]:
grouped_players = {
    pos: df_combined[df_combined["Flat_Positions"].apply(lambda x: pos in x)]
    for pos in numeric_cols
}

In [127]:
from torch.utils.data import DataLoader, TensorDataset


class PositionModelFactory:
    def __init__(self, numeric_cols, hidden_dim=128):
        self.models = {}
        self.numeric_cols = numeric_cols
        self.hidden_dim = hidden_dim

    def train_model_for_position(self, position, df_pos, scaler):
        # Prepare data
        X = df_pos[self.numeric_cols].values
        y = df_pos[["Rating", "Potential"]].values
        # Scale targets
        y_scaled = scaler.fit_transform(y)  # Or scaler.transform(y) if already fitted

        # Train/val split
        X_train, X_val, y_train, y_val = train_test_split(
            X, y_scaled, test_size=0.2, random_state=42
        )

        # Convert to tensors
        train_ds = TensorDataset(
            torch.tensor(X_train, dtype=torch.float32),
            torch.tensor(y_train, dtype=torch.float32),
        )
        val_ds = TensorDataset(
            torch.tensor(X_val, dtype=torch.float32),
            torch.tensor(y_val, dtype=torch.float32),
        )

        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=32)

        # Create model
        model = DeepRegressionModel(
            input_dim=len(self.numeric_cols), hidden_dim=self.hidden_dim
        )

        # Train loop (simplified)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()

        for epoch in range(10):  # Replace with more robust loop if needed
            model.train()
            for xb, yb in train_loader:
                preds = model(xb)
                loss = criterion(preds, yb)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        self.models[position] = model

In [128]:
factory = PositionModelFactory(numeric_cols)

for pos, df_pos in grouped_players.items():
    print(f"🔧 Training model for position: {pos}")
    factory.train_model_for_position(pos, df_pos, scaler_rp)

🔧 Training model for position: CAM
🔧 Training model for position: CD
🔧 Training model for position: CDM
🔧 Training model for position: CF
🔧 Training model for position: CM
🔧 Training model for position: DM
🔧 Training model for position: LAM
🔧 Training model for position: LD
🔧 Training model for position: LF
🔧 Training model for position: LM
🔧 Training model for position: RAM
🔧 Training model for position: RD
🔧 Training model for position: RF
🔧 Training model for position: RM


In [129]:
def evaluate_best_position_per_player(df, factory, numeric_cols, scaler_rp):
    predictions = []

    for _, row in df.iterrows():
        player_id = row["player_id"]
        raw_name = row["raw_name"]
        age = row["Age_Original"]
        features = np.array([row[col] for col in numeric_cols], dtype=np.float32)

        best_position = None
        best_rating = -np.inf
        best_potential = -np.inf  # Start low

        for pos in row["Flat_Positions"]:
            if pos not in factory.models:
                continue  # Skip positions without a trained model

            model = factory.models[pos]
            model.eval()

            with torch.no_grad():
                x = torch.tensor(features.reshape(1, -1), dtype=torch.float32)
                pred_scaled = model(x).cpu().numpy()
                pred_unscaled = scaler_rp.inverse_transform(pred_scaled)[
                    0
                ]  # [rating, potential]

            rating, potential = pred_unscaled

            if potential > best_potential:
                best_potential = potential
                best_rating = rating
                best_position = pos

        if best_position:
            predictions.append(
                {
                    "player_id": player_id,
                    "raw_name": raw_name,
                    "age": age,
                    "best_position": best_position,
                    "pred_rating": best_rating,
                    "pred_potential": best_potential,
                }
            )

    return pd.DataFrame(predictions)

In [130]:
pred_df = evaluate_best_position_per_player(
    df_combined, factory, numeric_cols, scaler_rp
)

print("\n🔝 Best Position-Based Predictions:\n")
for _, row in pred_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Rating: {row['pred_rating']:.2f} | Potential: {row['pred_potential']:.2f}"
    )


🔝 Best Position-Based Predictions:

Erling Haaland            | Age: 24.0 | Position: CF   | Rating: 64.07 | Potential: 64.84
Lamine Yamal              | Age: 17.0 | Position: RAM  | Rating: 3.86 | Potential: -2.67
Kylian Mbappé             | Age: 26.0 | Position: CF   | Rating: 64.07 | Potential: 64.84
Pedri                     | Age: 22.0 | Position: DM   | Rating: 68.94 | Potential: 66.27
Jude Bellingham           | Age: 22.0 | Position: CM   | Rating: 74.44 | Potential: 79.03
Florian Wirtz             | Age: 22.0 | Position: CAM  | Rating: 8.35 | Potential: 2.73
Pau Cubarsí               | Age: 18.0 | Position: CD   | Rating: 61.80 | Potential: 58.10
Jamal Musiala             | Age: 22.0 | Position: CM   | Rating: 59.61 | Potential: 61.95
Alexander Isak            | Age: 25.0 | Position: CF   | Rating: 64.07 | Potential: 64.84
Cole Palmer               | Age: 23.0 | Position: CM   | Rating: 59.61 | Potential: 61.95
Federico Valverde         | Age: 26.0 | Position: DM   | Rating: 7

In [131]:
def print_all_position_predictions(df, factory, numeric_cols, scaler_rp):
    for _, row in df.iterrows():
        raw_name = row["raw_name"]
        age = row["Age_Original"]
        features = np.array([row[col] for col in numeric_cols], dtype=np.float32)

        print(f"\n🧩 {raw_name} (Age: {age:.1f}) — Predictions by Position:")

        for pos in row["Flat_Positions"]:
            if pos not in factory.models:
                continue  # Skip untrained positions

            model = factory.models[pos]
            model.eval()

            with torch.no_grad():
                x = torch.tensor(features.reshape(1, -1), dtype=torch.float32)
                pred_scaled = model(x).cpu().numpy()
                pred_unscaled = scaler_rp.inverse_transform(pred_scaled)[0]

            rating, potential = pred_unscaled
            print(f"  📍 {pos:4} | Rating: {rating:.2f} | Potential: {potential:.2f}")

In [132]:
print_all_position_predictions(df_combined, factory, numeric_cols, scaler_rp)



🧩 Erling Haaland (Age: 24.0) — Predictions by Position:
  📍 CF   | Rating: 64.07 | Potential: 64.84

🧩 Lamine Yamal (Age: 17.0) — Predictions by Position:
  📍 RAM  | Rating: 3.86 | Potential: -2.67

🧩 Kylian Mbappé (Age: 26.0) — Predictions by Position:
  📍 CF   | Rating: 64.07 | Potential: 64.84

🧩 Pedri (Age: 22.0) — Predictions by Position:
  📍 CAM  | Rating: 9.25 | Potential: 3.80
  📍 DM   | Rating: 68.94 | Potential: 66.27

🧩 Jude Bellingham (Age: 22.0) — Predictions by Position:
  📍 CAM  | Rating: 10.72 | Potential: 5.36
  📍 CM   | Rating: 74.44 | Potential: 79.03
  📍 LM   | Rating: 64.97 | Potential: 65.31

🧩 Florian Wirtz (Age: 22.0) — Predictions by Position:
  📍 CAM  | Rating: 8.35 | Potential: 2.73

🧩 Pau Cubarsí (Age: 18.0) — Predictions by Position:
  📍 CD   | Rating: 61.80 | Potential: 58.10

🧩 Jamal Musiala (Age: 22.0) — Predictions by Position:
  📍 CM   | Rating: 59.61 | Potential: 61.95

🧩 Alexander Isak (Age: 25.0) — Predictions by Position:
  📍 CF   | Rating: 64.07 

# Model 2: because the previous one was not working


In [133]:
import torch
import torch.nn as nn


class PositionEmbedder(nn.Module):
    def __init__(self, num_positions, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_positions, embed_dim)

    def forward(self, indices):
        return self.embedding(indices)


class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, downsample=False):
        super().__init__()
        stride = 2 if downsample else 1
        self.residual = nn.Sequential(
            nn.Linear(in_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Linear(out_channels, out_channels),
            nn.BatchNorm1d(out_channels),
        )

        self.shortcut = nn.Sequential()
        if downsample or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Linear(in_channels, out_channels), nn.BatchNorm1d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        res = self.residual(x)
        shortcut = self.shortcut(x)
        return self.relu(res + shortcut)


class RatingPotentialModel(nn.Module):
    def __init__(
        self, input_dim, num_positions, hidden_dim=128, embed_dim=16, num_blocks=16
    ):
        super().__init__()
        self.pos_embedder = PositionEmbedder(num_positions, embed_dim)

        self.stem = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
        )

        blocks = []
        for i in range(num_blocks):
            downsample = i % 4 == 0 and i != 0
            in_channels = hidden_dim
            out_channels = hidden_dim
            blocks.append(
                BottleneckBlock(in_channels, out_channels, downsample=downsample)
            )
        self.resnet = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.Linear(hidden_dim + embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2),  # Output: rating, potential
        )

    def forward(self, x_num, position_indices):
        pos_embed = self.pos_embedder(position_indices)  # [batch_size, embed_dim]
        x = self.stem(x_num)  # [batch_size, hidden_dim]
        x = self.resnet(x)  # Residual pipeline
        x = torch.cat([x, pos_embed], dim=1)  # Merge numeric + positional
        out = self.head(x)
        return out

In [134]:
import torch
from torch.utils.data import Dataset


class PlayerPositionDataset(Dataset):
    def __init__(self, X_num, X_pos, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_pos = torch.tensor(X_pos, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_pos[idx], self.y[idx]

In [135]:
def train_model(
    model,
    train_loader,
    val_loader,
    max_epochs=500,
    patience=20,
    min_delta=1e-2,
    lr=1e-5,
):
    import torch.nn as nn

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_epoch = 0
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0

        for X_num, X_pos, y in train_loader:
            X_num, X_pos, y = X_num.to(device), X_pos.to(device), y.to(device)
            preds = model(X_num, X_pos)
            loss = criterion(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_num, X_pos, y in val_loader:
                X_num, X_pos, y = X_num.to(device), X_pos.to(device), y.to(device)
                preds = model(X_num, X_pos)
                loss = criterion(preds, y)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        history.append((epoch, avg_train_loss, avg_val_loss))
        print(
            f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}"
        )

        if avg_val_loss + min_delta < best_val_loss:
            best_val_loss = avg_val_loss
            best_epoch = epoch
            wait = 0
            # Optional model saving
            # torch.save(model.state_dict(), f"best_model_{best_epoch}.pt")
        else:
            wait += 1
            if wait >= patience:
                print(f"🛑 Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
                break

    return history

In [136]:
def train_models_per_position(expanded_df, numeric_cols, position_vocab):
    models = {}
    pos_idx = 0
    for position in numeric_cols:
        subset = expanded_df[expanded_df["position"] == position]
        subset = subset.dropna(subset=["rating", "potential"])

        if len(subset) < 50:
            print(f"⏭️ Skipping position '{position}' — only {len(subset)} players")
            continue

        # Feature scaling
        scaler_X = StandardScaler()
        X_numeric = scaler_X.fit_transform(subset[numeric_cols].values)

        # Target scaling (POSITION-SPECIFIC)
        y = subset[["rating", "potential"]].values
        scaler_y = MinMaxScaler(feature_range=(0, 100))
        y_scaled = scaler_y.fit_transform(y)

        X_position = np.full(len(subset), pos_idx)

        # Train/val split
        X_num_train, X_num_val, X_pos_train, X_pos_val, y_train, y_val = (
            train_test_split(
                X_numeric, X_position, y_scaled, test_size=0.2, random_state=42
            )
        )

        # Loaders
        train_dataset = PlayerPositionDataset(X_num_train, X_pos_train, y_train)
        val_dataset = PlayerPositionDataset(X_num_val, X_pos_val, y_val)
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=32, shuffle=True
        )
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32)
        pos_idx += 1
        # Build and train model
        model = RatingPotentialModel(
            input_dim=X_numeric.shape[1], num_positions=len(position_vocab)
        )

        print(f"\n🟢 Training model for position: {position} (Players: {len(subset)})")

        try:
            history = train_model(model, train_loader, val_loader)
            models[position] = {
                "model": model,
                "scaler": scaler_y,  # ✅ attach position-specific scaler
            }
        except Exception as e:
            print(f"❌ Error training model for '{position}': {e}")
            continue

    return models

In [137]:
models = train_models_per_position(expanded_df, numeric_cols, position_vocab)


🟢 Training model for position: CAM (Players: 164)
Epoch 1: Train Loss=1877.7129, Val Loss=1319.8177
Epoch 2: Train Loss=1762.6647, Val Loss=1319.6105
Epoch 3: Train Loss=2338.3962, Val Loss=1318.5057
Epoch 4: Train Loss=1822.6083, Val Loss=1319.1935
Epoch 5: Train Loss=1627.4943, Val Loss=1320.4989
Epoch 6: Train Loss=1752.0984, Val Loss=1317.3787
Epoch 7: Train Loss=1633.4896, Val Loss=1315.6008
Epoch 8: Train Loss=1651.4818, Val Loss=1308.7686
Epoch 9: Train Loss=1841.9880, Val Loss=1296.1619
Epoch 10: Train Loss=1849.1474, Val Loss=1301.0444
Epoch 11: Train Loss=1672.6568, Val Loss=1289.0335
Epoch 12: Train Loss=2298.9635, Val Loss=1296.0078
Epoch 13: Train Loss=1672.4330, Val Loss=1295.9777
Epoch 14: Train Loss=1693.3108, Val Loss=nan
Epoch 15: Train Loss=nan, Val Loss=nan
Epoch 16: Train Loss=nan, Val Loss=nan
Epoch 17: Train Loss=nan, Val Loss=nan
Epoch 18: Train Loss=nan, Val Loss=nan
Epoch 19: Train Loss=nan, Val Loss=nan
Epoch 20: Train Loss=nan, Val Loss=nan
Epoch 21: Train 

In [138]:
import torch


def evaluate_best_position_for_player(models, player_row, numeric_cols, position_vocab):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    predictions = []

    # Extract and validate player features
    try:
        X_numeric = player_row[numeric_cols].values.astype(np.float32)
    except KeyError as e:
        print(f"⚠️ Missing numeric column: {e}")
        return None

    positions = player_row.get("Flat_Positions", [])
    if not isinstance(positions, list) or not positions:
        print(f"⚠️ Player '{player_row.get('raw_name', '')}' has no valid positions")
        return None

    for pos in positions:
        entry = models.get(pos)
        pos_idx = position_vocab.get(pos)

        if entry is None:
            print(f"🔍 No trained model for position: {pos}")
            continue
        if pos_idx is None:
            print(f"❓ Position '{pos}' not found in vocabulary")
            continue

        model = entry["model"]
        scaler_y = entry["scaler"]

        model.to(device)
        model.eval()

        # Forward pass
        with torch.no_grad():
            X_num_tensor = torch.tensor([X_numeric], dtype=torch.float32).to(device)
            X_pos_tensor = torch.tensor([pos_idx], dtype=torch.long).to(device)
            pred_scaled = model(X_num_tensor, X_pos_tensor).cpu().numpy()

            try:
                pred_unscaled = scaler_y.inverse_transform(pred_scaled)[0]
            except ValueError as e:
                print(f"❌ Inverse scaling failed for position '{pos}': {e}")
                continue

        predictions.append(
            {
                "position": pos,
                "rating": float(pred_unscaled[0]),
                "potential": float(pred_unscaled[1]),
            }
        )

    if not predictions:
        print(
            f"⚠️ No predictions generated for player '{player_row.get('raw_name', '')}'"
        )
        return None

    # Select best position based on highest potential
    best_position = max(predictions, key=lambda x: x["potential"])

    return {
        "best_position": best_position["position"],
        "pred_rating": best_position["rating"],
        "pred_potential": best_position["potential"],
        "all_predictions": predictions,
    }

In [139]:
def print_best_position_predictions(predictions_df):
    if predictions_df.empty:
        print("⚠️ No position-based predictions available.")
        return

    # Header
    print("\n🔝 Best Position-Based Predictions:\n")
    print(
        f"{'Player Name':<25} | {'Age':<5} | {'Pos':<6} | {'Rating':<7} | {'Potential':<9}"
    )
    print("-" * 65)

    for _, row in predictions_df.iterrows():
        name = str(row.get("raw_name", "N/A"))[:25]  # Truncate long names
        age = float(row.get("age", 0.0))  # Ensure float for formatting
        position = str(row.get("best_position", "??"))  # Default fallback
        rating = float(row.get("pred_rating", 0.0))
        potential = float(row.get("pred_potential", 0.0))

        print(
            f"{name:<25} | {age:>4.1f} | {position:<6} | {rating:>6.2f} | {potential:>8.2f}"
        )

In [140]:
results = []

for _, row in df_combined.iterrows():
    pred = evaluate_best_position_for_player(
        models,
        row,
        numeric_cols,
        position_vocab,  # ✅ Removed scaler_rp
    )
    if pred:
        pred["raw_name"] = row.get("raw_name", "N/A")
        pred["age"] = row.get("Age_Original", None)
        results.append(pred)

predictions_df = pd.DataFrame(results)
print_best_position_predictions(predictions_df)

🔍 No trained model for position: F
🔍 No trained model for position: AM
🔍 No trained model for position: M
🔍 No trained model for position: F
🔍 No trained model for position: AM
🔍 No trained model for position: M
🔍 No trained model for position: AM
🔍 No trained model for position: M
🔍 No trained model for position: AM
🔍 No trained model for position: D
🔍 No trained model for position: M
🔍 No trained model for position: F
🔍 No trained model for position: M
🔍 No trained model for position: D
🔍 No trained model for position: M
🔍 No trained model for position: AM
🔍 No trained model for position: M
🔍 No trained model for position: M
🔍 No trained model for position: F
🔍 No trained model for position: M
🔍 No trained model for position: F
🔍 No trained model for position: AM
🔍 No trained model for position: F
🔍 No trained model for position: M
🔍 No trained model for position: F
🔍 No trained model for position: M
🔍 No trained model for position: F
🔍 No trained model for position: M
🔍 No trained m